# Lab 7: Spiking Neural Network on a Benchmark (N-MNIST)

**Name:** Anson Dias

## Objective
1. Implement a **three-layer spiking network** in **Brian2**:
   * **Input encoding layer**: rate coding, each pixel's Poisson spike rate is proportional to its intensity.
   * **Hidden recurrent layer**: LIF neurons with adaptive thresholds and **recurrent** connections
     (fixed lateral inhibition plus plastic recurrent excitation). Input→hidden and hidden→hidden weights learn with **online STDP**.
   * **Output winner-take-all (WTA) layer**: 10 LIF neurons (one per digit) with mutual inhibition.
     Hidden→output weights learn with STDP while a teacher signal drives the correct label neuron.
2. Train on **N-MNIST** (MNIST recorded with a Dynamic Vision Sensor) for **20 epochs** of online STDP.
3. Measure classification accuracy on a **500-image test subset**.
4. Compare **total spike count** (an energy proxy) with an **equivalent ANN trained in PyTorch**.
5. Plot **accuracy vs spike count** to show the energy-accuracy trade-off.

> **Runtime.** About 30–60 min on a free Google Colab CPU runtime with the default settings in
> the config cell (2,000 training images × 20 epochs). Set `QUICK = True` for a few-minute smoke test.

## 0. Setup
Brian2 is not preinstalled on Colab. PyTorch, NumPy and Matplotlib are.

In [ ]:
%pip install -q brian2 cython

In [ ]:
import os, io, json, time, zipfile, hashlib, urllib.request
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import brian2 as b2
from brian2 import ms, mV, Hz, second
import torch
import torch.nn as nn

b2.prefs.codegen.target = "cython"   # compiled C code via Cython (needs a C compiler; Colab has one)
b2.prefs.logging.std_redirection = False
b2.BrianLogger.log_level_error()     # hide Brian2's informational warnings

# Chart style: one fixed categorical order (blue, orange, aqua), single-hue maps, recessive axes
BLUE, ORANGE, AQUA, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
                     "lines.markersize": 6, "grid.alpha": 0.25, "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA])})

print("brian2", b2.__version__, "| torch", torch.__version__, "| numpy", np.__version__)

### Experiment configuration
Every number reported below comes from this cell. The task fixes the epochs (20) and the test-set size (500).
The training subset is balanced (equal images per digit) and the same images are reused every epoch,
shuffled each time.

In [ ]:
QUICK = False   # True = small, fast smoke test (do not submit QUICK results)

CFG = dict(
    data_dir="./data/nmnist",  # where train.zip / test.zip are (or will be downloaded)
    seed=7,
    n_train=2000,              # training images per epoch (200 per digit)
    n_test=500,                # test subset, 50 per digit (required by the task)
    epochs=20,                 # online-STDP epochs (required by the task)
    n_hidden=100,              # hidden recurrent layer size
    dt_ms=0.5,                 # simulation time step
    t_present_ms=100.0,        # presentation time per image
    max_rate_hz=100.0,         # rate code: rate = intensity * max_rate
    chunk=50,                  # images simulated per Brian2 run() call
)
if QUICK:
    CFG.update(n_train=200, n_test=100, epochs=3)

rng = np.random.default_rng(CFG["seed"])
torch.manual_seed(CFG["seed"])
print(json.dumps(CFG, indent=1))

## 1. The N-MNIST dataset

N-MNIST (Orchard et al., 2015) was made by putting an ATIS event camera on a pan-tilt unit and moving it in
**three saccades** (about 100 ms each) in front of MNIST digits shown on a monitor. Each sample is a list of
events `(x, y, polarity, timestamp)` on a **34 × 34** sensor. Polarity 1 means brightness went up (ON),
0 means it went down (OFF).

Every `.bin` file stores 5 bytes per event:

| bits | 0–7 | 8–15 | 16 | 17–39 |
|---|---|---|---|---|
| field | x | y | polarity | timestamp (µs) |

The code below downloads the official archives from Mendeley Data (the URLs used by the `tonic` library) and
reads the binary files straight from the zip, so nothing needs to be extracted.
If the download is blocked, download **Train.zip** and **Test.zip** by hand from
<https://www.garrickorchard.com/datasets/n-mnist> (or doi:10.17632/468j46mzdv.1) and put them in `CFG["data_dir"]`.
Extracted `Train/` and `Test/` folders also work.

In [ ]:
NMNIST_FILES = {
    "train": dict(url="https://data.mendeley.com/public-files/datasets/468j46mzdv/files/"
                      "39c25547-014b-4137-a934-9d29fa53c7a0/file_downloaded",
                  names=("train.zip", "Train.zip", "Train"), md5="20959b8e626244a1b502305a9e6e2031"),
    "test":  dict(url="https://data.mendeley.com/public-files/datasets/468j46mzdv/files/"
                      "05a4d654-7e03-4c15-bdfa-9bb2bcbea494/file_downloaded",
                  names=("test.zip", "Test.zip", "Test"), md5="69ca8762b2fe404d9b9bad1103e97832"),
}
SENSOR = (34, 34)
N_IN = SENSOR[0] * SENSOR[1]          # 1156 input neurons (ON and OFF events merged)
N_OUT = 10


def md5sum(path, block=1 << 22):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def locate_or_download(split, data_dir):
    """Return a path to the split's zip file or extracted folder, downloading it if needed."""
    info = NMNIST_FILES[split]
    os.makedirs(data_dir, exist_ok=True)
    for name in info["names"]:
        p = os.path.join(data_dir, name)
        if os.path.exists(p):
            return p
    dest = os.path.join(data_dir, info["names"][0])
    print(f"Downloading N-MNIST {split} split (train is about 1 GB) ...")

    def hook(n, bs, total):
        if total > 0 and n % 500 == 0:
            print(f"\r  {min(n * bs, total) / 1e6:7.1f} / {total / 1e6:.1f} MB", end="")

    urllib.request.urlretrieve(info["url"], dest + ".part", reporthook=hook)
    os.replace(dest + ".part", dest)
    print()
    if md5sum(dest) != info["md5"]:
        print("  WARNING: MD5 mismatch, the file may be corrupt")
    return dest


class NMNISTSplit:
    """Read-only access to one N-MNIST split, from a zip archive or an extracted folder."""

    def __init__(self, path):
        self.path, self.zip = path, None
        if os.path.isdir(path):
            members = [os.path.relpath(os.path.join(r, f), path)
                       for r, _, fs in os.walk(path) for f in fs]
        else:
            self.zip = zipfile.ZipFile(path)
            members = self.zip.namelist()
        self.items = []
        for m in sorted(members):
            parts = m.replace("\\", "/").split("/")
            if m.endswith(".bin") and "__MACOSX" not in parts and parts[-2].isdigit():
                self.items.append((m, int(parts[-2])))
        self.labels = np.array([lab for _, lab in self.items])
        if not self.items:
            raise RuntimeError(f"No N-MNIST .bin files found in {path}. Expected <digit>/<id>.bin entries, "
                               "e.g. Train/3/00010.bin. Check that the archive downloaded completely.")

    def __len__(self):
        return len(self.items)

    def raw(self, k):
        m = self.items[k][0]
        if self.zip is not None:
            return self.zip.read(m)
        with open(os.path.join(self.path, m), "rb") as f:
            return f.read()

    def events(self, k):
        return decode_events(self.raw(k))


def decode_events(raw):
    """Decode the 40-bit N-MNIST event format into x, y, polarity, t (µs) arrays."""
    b = np.frombuffer(raw, dtype=np.uint8).astype(np.uint32)
    b = b[: (len(b) // 5) * 5]
    x, y = b[0::5], b[1::5]
    p = (b[2::5] >> 7) & 1
    t = ((b[2::5] & 127) << 16) | (b[3::5] << 8) | b[4::5]
    return x, y, p, t


def balanced_subset(labels, n, rng):
    per_class = n // N_OUT
    idx = np.concatenate([rng.choice(np.flatnonzero(labels == c), per_class, replace=False)
                          for c in range(N_OUT)])
    return rng.permutation(idx)


train_split = NMNISTSplit(locate_or_download("train", CFG["data_dir"]))
test_split = NMNISTSplit(locate_or_download("test", CFG["data_dir"]))
print(f"N-MNIST: {len(train_split)} training samples, {len(test_split)} test samples")
print("class counts (train):", np.bincount(train_split.labels, minlength=10))

### 1.1 Looking at the raw events
One sample, plotted three ways: events in (x, y) colored by polarity, the event rate over the
~300 ms recording (three saccades of ~100 ms), and events in (t, x) space, where the sensor's motion is visible.

In [ ]:
x, y, p, t = train_split.events(0)
lab0 = train_split.items[0][1]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(x[p == 1], y[p == 1], s=2, c=ORANGE, label="ON")
ax[0].scatter(x[p == 0], y[p == 0], s=2, c=BLUE, label="OFF")
ax[0].set(xlim=(-0.5, 33.5), ylim=(33.5, -0.5), aspect="equal",
          title=f"Sample 0 (digit {lab0}): {len(t)} events")
ax[0].legend(loc="lower right", markerscale=4)
ax[1].hist(t / 1000, bins=np.arange(0, t.max() / 1000 + 3, 3), color=BLUE)
ax[1].set(xlabel="time (ms)", ylabel="events per 3 ms bin", title="Event rate over the three saccades")
ax[2].scatter(t / 1000, x, s=1, c=np.where(p == 1, ORANGE, BLUE))
ax[2].set(xlabel="time (ms)", ylabel="x", title="Events in (t, x) space")
plt.tight_layout(); plt.show()

## 2. Preprocessing and rate coding

**Event frame → intensity.** All events of a sample (both polarities, all three saccades) are added up per
pixel into a 34 × 34 count image *C*. It is scaled to an intensity in [0, 1] using the image's
99th-percentile count, which keeps a few hot pixels from setting the scale:

$$I_i = \min\left(1,\ \frac{C_i}{\mathrm{P}_{99}(C)}\right)$$

**Rate coding.** Input neuron *i* fires as a Poisson process with rate $r_i = I_i \cdot r_{max}$, so
**spike rate is proportional to pixel intensity**. In each time step of length $\Delta t$, neuron *i* spikes
with probability $r_i \Delta t$. That is the same process `brian2.PoissonGroup` generates. The spikes are
drawn with NumPy and fed through a `SpikeGeneratorGroup` for two reasons: many images can then be simulated
in one `run()` call (Brian2's per-call overhead is about 0.25 s), and the exact number of input spikes is
known for the energy analysis.

In [ ]:
def events_to_intensity(x, y):
    keep = (x < SENSOR[1]) & (y < SENSOR[0])
    counts = np.bincount(y[keep] * SENSOR[1] + x[keep], minlength=N_IN).astype(np.float32)
    scale = max(np.percentile(counts, 99), 1.0)
    return np.clip(counts / scale, 0.0, 1.0)


def build_frames(split, idx):
    X = np.stack([events_to_intensity(*split.events(k)[:2]) for k in idx])
    n_events = np.array([len(split.raw(k)) // 5 for k in idx])
    return X, split.labels[idx], n_events


t0 = time.time()
tr_idx = balanced_subset(train_split.labels, CFG["n_train"], rng)
te_idx = balanced_subset(test_split.labels, CFG["n_test"], rng)
X_train, y_train, ev_train = build_frames(train_split, tr_idx)
X_test, y_test, ev_test = build_frames(test_split, te_idx)
print(f"train frames {X_train.shape}, test frames {X_test.shape}  ({time.time() - t0:.1f}s)")
print(f"events per sample: mean {ev_train.mean():.0f}, min {ev_train.min()}, max {ev_train.max()}")
print(f"mean intensity {X_train.mean():.3f} -> mean input rate per image "
      f"{X_train.sum(1).mean() * CFG['max_rate_hz']:.0f} Hz (summed over {N_IN} inputs)")


def poisson_spikes(X, max_rate_hz, T_ms, dt_ms, rng, t_offset_ms=0.0):
    """Rate-code a batch of images; image k occupies [t_offset + k*T, t_offset + (k+1)*T).
    Returns (neuron indices, spike times in ms, spikes per image)."""
    n_steps = int(round(T_ms / dt_ms))
    p = (X * max_rate_hz * dt_ms * 1e-3).astype(np.float32)
    ids, times, per_img = [], [], np.zeros(len(X), dtype=np.int64)
    for k in range(len(X)):
        step, i = np.nonzero(rng.random((n_steps, X.shape[1]), dtype=np.float32) < p[k])
        ids.append(i)
        times.append(t_offset_ms + (k * n_steps + step) * dt_ms)
        per_img[k] = len(i)
    return np.concatenate(ids), np.concatenate(times), per_img


fig, ax = plt.subplots(2, 10, figsize=(16, 3.8))
for c in range(10):
    k = np.flatnonzero(y_train == c)[0]
    ax[0, c].imshow(X_train[k].reshape(SENSOR), cmap="Greys", vmin=0, vmax=1)
    ax[0, c].set_title(f"digit {c}"); ax[0, c].axis("off")
    k = np.flatnonzero(y_test == c)[0]
    ax[1, c].imshow(X_test[k].reshape(SENSOR), cmap="Greys", vmin=0, vmax=1); ax[1, c].axis("off")
ax[0, 0].text(-12, 17, "train", rotation=90, va="center"); ax[1, 0].text(-12, 17, "test", rotation=90, va="center")
plt.suptitle("N-MNIST event frames scaled to intensity (input to the rate encoder)"); plt.tight_layout(); plt.show()

ids, ts, _ = poisson_spikes(X_train[:1], CFG["max_rate_hz"], CFG["t_present_ms"], CFG["dt_ms"], rng)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw=dict(width_ratios=[3, 1]))
ax[0].plot(ts, ids, "|", c=INK, ms=2)
ax[0].set(xlabel="time (ms)", ylabel="input neuron", title=f"Rate-coded input, {len(ids)} spikes in {CFG['t_present_ms']:.0f} ms")
counts = np.bincount(ids, minlength=N_IN)
ax[1].scatter(X_train[0] * CFG["max_rate_hz"], counts / (CFG["t_present_ms"] / 1000), s=4, alpha=.5)
ax[1].plot([0, CFG["max_rate_hz"]], [0, CFG["max_rate_hz"]], "--", c=GRAY, lw=1)
ax[1].set(xlabel="target rate (Hz)", ylabel="measured rate (Hz)", title="rate ∝ intensity")
plt.tight_layout(); plt.show()

## 3. The three-layer spiking network (Brian2)

```
                         recurrent: lateral inhibition (fixed)
                                    + sparse excitation (STDP)
                                       ┌──────┐
                                       ▼      │
 Input layer      ──STDP──►     Hidden layer  ──STDP──►   Output WTA layer
 1156 Poisson                   100 LIF, adaptive θ        10 LIF + lateral inhibition
 (rate code)                                                   ▲
                                                   Teacher spikes (training only)
```

**Neurons** are conductance-based leaky integrate-and-fire units, as in Diehl & Cook (2015):

$$\tau_m \frac{dv}{dt} = (v_{rest}-v) + g_e (E_{exc}-v) + g_i (E_{inh}-v), \qquad
  \tau_e \frac{dg_e}{dt} = -g_e, \qquad \tau_i \frac{dg_i}{dt} = -g_i$$

A spike occurs when $v > v_{thr} + \theta$. In the hidden layer, $\theta$ is an adaptive threshold
(homeostasis): it grows by $\theta_+$ with each spike and decays with time constant $\tau_\theta$
(200 s, about one epoch). A neuron that wins often gets harder to excite, so all 100 neurons end up being used.

**Hidden recurrent layer.**
* *Lateral inhibition*: every hidden spike adds $w_{inh}$ to the inhibitory conductance of all other
  hidden neurons. This is soft winner-take-all competition, so different neurons learn different digit prototypes.
* *Recurrent excitation*: sparse ($p = 0.1$) hidden→hidden excitatory synapses, also learned with STDP.

**Online STDP.** Each synapse keeps a presynaptic trace $x_{pre}$ and a postsynaptic trace $x_{post}$
(both set to 1 at a spike, decaying with $\tau = 20$ ms). Every spike changes the weight immediately
(a simplified form of the Diehl & Cook 2015 rule with a presynaptic offset):

* presynaptic spike: $\;w \leftarrow w - \eta_{pre}\, x_{post}$ (depression: input arrived after the neuron fired)
* postsynaptic spike: $\;w \leftarrow w + \eta_{post}\,(x_{pre} - x_{tar})$ (input that fired just before
  the neuron is strengthened, and inputs that were silent ($x_{pre} < x_{tar}$) are weakened)

Weights are clipped to $[0, w_{max}]$.

**Synaptic scaling (L2).** After every chunk of 50 training images, each neuron's incoming weight vector
$\mathbf{w}_k$ is rescaled to a fixed **L2 norm**. The input drive is then
$\mathbf{x}\cdot\mathbf{w}_k = \lVert\mathbf{x}\rVert\,\lVert\mathbf{w}_k\rVert\cos\phi_k$, and since every neuron has the same
$\lVert\mathbf{w}_k\rVert$, the neuron whose prototype has the highest **cosine similarity** to the image wins.
Diehl & Cook fixed the *sum* of the weights (L1) instead. On the dense, motion-blurred DVS frames that favours
neurons whose weights are concentrated on the centre of the sensor, which match almost any digit. L2 scaling avoids that bias.

**Output WTA layer with a teacher.** During training, a teacher spike train (Poisson, 100 Hz) drives the output
neuron of the correct label, and the hidden→output synapses do not transmit. Output spikes are then purely
teacher-driven, and STDP strengthens the synapses from hidden neurons that fire for that digit.
During testing the teacher is off, the synapses transmit, plasticity is frozen, and the 10 output neurons
compete through lateral inhibition. **The prediction is the output neuron with the most spikes.** If no
output neuron spikes, or two tie, the total synaptic input each output neuron received breaks the tie.

Every state variable (membrane potential, conductances, STDP traces) is reset at the start of each image,
so images don't leak into each other.

In [ ]:
P = dict(
    # neuron constants (Diehl & Cook 2015)
    v_rest=-65 * mV, v_reset=-65 * mV, E_exc=0 * mV, E_inh=-100 * mV,
    tau_ge=1 * ms, tau_gi=2 * ms,
    # hidden layer
    tau_m_h=100 * ms, v_thr_h=-52 * mV, refrac_h=5 * ms,
    theta_plus=0.05 * mV, tau_theta=200 * second,   # adaptive threshold (homeostasis)
    w_inh_h=40.0,                 # recurrent lateral inhibition
    p_rec=0.1,                    # recurrent excitation (connection probability)
    # output layer
    tau_m_o=100 * ms, v_thr_o=-52 * mV, refrac_o=2 * ms,
    w_inh_o=20.0, gain_o=25.0,    # output lateral inhibition and hidden->output gain
    teacher_rate=100 * Hz, w_teach=50.0,
    # STDP traces
    tau_pre=20 * ms, tau_post=20 * ms,
    # STDP per pathway: input->hidden (ih), hidden->hidden recurrent (hh), hidden->output (ho)
    nu_pre_ih=1e-4, nu_post_ih=1e-3, x_tar_ih=0.1, wmax_ih=1.0,
    nu_pre_hh=1e-4, nu_post_hh=1e-3, x_tar_hh=0.1, wmax_hh=0.5,
    nu_pre_ho=1e-4, nu_post_ho=5e-3, x_tar_ho=0.05, wmax_ho=1.0,
    # synaptic scaling targets (L2 norm of each neuron's incoming weight vector)
    norm_ih=2.5, norm_ho=1.5,
)

STDP_MODEL = """
w : 1
lr : 1 (shared)
dpre/dt = -pre/tau_pre : 1 (event-driven)
dpost/dt = -post/tau_post : 1 (event-driven)
"""
STDP_PRE = "pre = 1.; w = clip(w - lr*nu_pre*post, 0, wmax)"
STDP_POST = "w = clip(w + lr*nu_post*(pre - x_tar), 0, wmax); post = 1."


def stdp_ns(ns, tag):
    """Namespace for one plastic pathway: the generic STDP names mapped to that pathway's constants."""
    return dict(ns, **{k: ns[f"{k}_{tag}"] for k in ("nu_pre", "nu_post", "x_tar", "wmax")})

LIF_EQS = """
dv/dt = ((v_rest - v) + ge*(E_exc - v) + gi*(E_inh - v)) / tau_m : volt (unless refractory)
dge/dt = -ge/tau_ge : 1
dgi/dt = -gi/tau_gi : 1
"""


class SpikingNet:
    def __init__(self, n_in, n_hidden, n_out, P, cfg, seed):
        b2.start_scope()
        b2.seed(seed)
        b2.defaultclock.dt = cfg["dt_ms"] * ms
        self.P, self.cfg, self.nh = P, cfg, n_hidden
        self.rng = np.random.default_rng(seed)
        ns = {k: v for k, v in P.items()}

        # Layer 1: input encoding (rate-coded Poisson spikes) and the teacher signal
        self.inp = b2.SpikeGeneratorGroup(n_in, [], [] * ms, name="input")
        self.teacher = b2.SpikeGeneratorGroup(n_out, [], [] * ms, name="teacher")

        # Layer 2: hidden recurrent layer, LIF with adaptive threshold
        self.hid = b2.NeuronGroup(
            n_hidden, LIF_EQS + "dtheta/dt = -plastic*theta/tau_theta : volt\nplastic : 1 (shared)",
            threshold="v > v_thr_h + theta", reset="v = v_reset; theta += plastic*theta_plus",
            refractory=P["refrac_h"], method="exponential_euler", name="hidden",
            namespace=dict(ns, tau_m=P["tau_m_h"]))
        self.hid.v = P["v_rest"]

        # Layer 3: output winner-take-all layer
        self.out = b2.NeuronGroup(
            n_out, LIF_EQS, threshold="v > v_thr_o", reset="v = v_reset",
            refractory=P["refrac_o"], method="exponential_euler", name="output",
            namespace=dict(ns, tau_m=P["tau_m_o"]))
        self.out.v = P["v_rest"]

        # input -> hidden (plastic, all-to-all)
        self.S_ih = b2.Synapses(self.inp, self.hid, STDP_MODEL, on_pre="ge_post += w; " + STDP_PRE,
                                on_post=STDP_POST, name="S_ih", namespace=stdp_ns(ns, "ih"))
        self.S_ih.connect()
        self.S_ih.w = "0.3*rand()"
        # hidden -> hidden recurrent excitation (plastic, sparse)
        self.S_hh = b2.Synapses(self.hid, self.hid, STDP_MODEL, on_pre="ge_post += w; " + STDP_PRE,
                                on_post=STDP_POST, name="S_hh", namespace=stdp_ns(ns, "hh"))
        self.S_hh.connect(condition="i != j", p=P["p_rec"])
        self.S_hh.w = "0.1*rand()"
        # hidden -> hidden recurrent lateral inhibition (fixed)
        self.S_inh_h = b2.Synapses(self.hid, self.hid, on_pre="gi_post += w_inh_h", name="S_inh_h", namespace=ns)
        self.S_inh_h.connect(condition="i != j")
        # hidden -> output (plastic; transmits only at test time)
        self.S_ho = b2.Synapses(self.hid, self.out, STDP_MODEL + "transmit : 1 (shared)",
                                on_pre="ge_post += transmit*gain_o*w; " + STDP_PRE,
                                on_post=STDP_POST, name="S_ho", namespace=stdp_ns(ns, "ho"))
        self.S_ho.connect()
        self.S_ho.w = "0.1*rand()"
        # output lateral inhibition (WTA) and teacher -> output (one-to-one)
        self.S_inh_o = b2.Synapses(self.out, self.out, on_pre="gi_post += w_inh_o", name="S_inh_o", namespace=ns)
        self.S_inh_o.connect(condition="i != j")
        self.S_teach = b2.Synapses(self.teacher, self.out, on_pre="ge_post += w_teach", name="S_teach", namespace=ns)
        self.S_teach.connect(j="i")

        self.mon_h = b2.SpikeMonitor(self.hid, name="mon_hidden")
        self.mon_o = b2.SpikeMonitor(self.out, name="mon_output")
        self.net = b2.Network(self.inp, self.teacher, self.hid, self.out, self.S_ih, self.S_hh,
                              self.S_inh_h, self.S_ho, self.S_inh_o, self.S_teach, self.mon_h, self.mon_o)
        self._resetters = {}
        self._seen_h = self._seen_o = 0
        # post-synaptic indices, used for synaptic scaling
        self._j_ih, self._j_ho = np.asarray(self.S_ih.j[:]), np.asarray(self.S_ho.j[:])
        self.normalize()

    # fan-out per neuron, used to count synaptic operations
    def fanout(self):
        return dict(input=self.nh,
                    hidden=(len(self.S_hh) + len(self.S_inh_h) + len(self.S_ho)) / self.nh,
                    output=N_OUT - 1)

    def normalize(self):
        """Synaptic scaling: rescale each neuron's incoming weight vector to a fixed L2 norm."""
        for S, j, target in ((self.S_ih, self._j_ih, self.P["norm_ih"]),
                             (self.S_ho, self._j_ho, self.P["norm_ho"])):
            w = np.asarray(S.w[:])
            size = np.sqrt(np.bincount(j, weights=w ** 2, minlength=S.target.N))
            S.w[:] = w * (target / np.maximum(size, 1e-12))[j]

    def _resetters_for(self, T):
        """Reset every state variable at the start of each image (period T)."""
        key = round(float(T / ms), 6)
        if key not in self._resetters:
            objs = [g.run_regularly("v = v_rest; ge = 0; gi = 0", dt=T, when="start", name=f"{g.name}_reset_{len(self._resetters)}")
                    for g in (self.hid, self.out)]
            objs += [S.run_regularly("pre = 0; post = 0", dt=T, when="start", name=f"{S.name}_reset_{len(self._resetters)}")
                     for S in (self.S_ih, self.S_hh, self.S_ho)]
            self.net.add(*objs)
            self._resetters[key] = objs
        for k, objs in self._resetters.items():
            for o in objs:
                o.active = (k == key)

    def _new_spikes(self, mon, seen):
        i, t = np.asarray(mon.i[seen:]), np.asarray(mon.t[seen:] / ms)
        return i, t, mon.num_spikes

    def present(self, X, y=None, train=False, T_ms=None, max_rate_hz=None, lr=1.0, verbose=False):
        """Show every image in X once (lr scales the STDP learning rates when training).
        Returns per-image spike counts {input, hidden, output}."""
        cfg = self.cfg
        T_ms = T_ms or cfg["t_present_ms"]
        max_rate_hz = max_rate_hz or cfg["max_rate_hz"]
        T, dt = T_ms * ms, cfg["dt_ms"]
        # plasticity switches (shared variables, so no recompilation)
        self.hid.plastic = float(train)
        for S in (self.S_ih, self.S_hh, self.S_ho):
            S.lr = float(lr) if train else 0.0
        self.S_ho.transmit = 0.0 if train else 1.0

        # line up the phase start with the reset clock (idle period with no input)
        t_now = float(self.net.t / ms)
        t_start = np.ceil(t_now / T_ms - 1e-9) * T_ms
        for objs in self._resetters.values():
            for o in objs:
                o.active = False
        if t_start - t_now > 1e-9:
            self.inp.set_spikes([], [] * ms); self.teacher.set_spikes([], [] * ms)
            self.net.run((t_start - t_now) * ms)
        self._resetters_for(T)
        self._seen_h = self.mon_h.num_spikes
        self._seen_o = self.mon_o.num_spikes

        n = len(X)
        counts = dict(input=np.zeros(n, np.int64), hidden=np.zeros((n, self.nh), np.int64),
                      output=np.zeros((n, N_OUT), np.int64))
        for c0 in range(0, n, cfg["chunk"]):
            c1 = min(n, c0 + cfg["chunk"])
            t_off = float(self.net.t / ms)
            ids, ts, per_img = poisson_spikes(X[c0:c1], max_rate_hz, T_ms, dt, self.rng, t_off)
            self.inp.set_spikes(ids, ts * ms)
            counts["input"][c0:c1] = per_img
            if train and y is not None:   # Poisson teacher on the label neuron
                n_steps = int(round(T_ms / dt))
                p_t = float(self.P["teacher_rate"] * dt * ms)
                tid, tts = [], []
                for k, lab in enumerate(y[c0:c1]):
                    st = np.flatnonzero(self.rng.random(n_steps) < p_t)
                    tid.append(np.full(len(st), lab)); tts.append(t_off + (k * n_steps + st) * dt)
                self.teacher.set_spikes(np.concatenate(tid), np.concatenate(tts) * ms)
            else:
                self.teacher.set_spikes([], [] * ms)
            self.net.run((c1 - c0) * T)
            for mon, key, attr in ((self.mon_h, "hidden", "_seen_h"), (self.mon_o, "output", "_seen_o")):
                seen = getattr(self, attr)
                i, t, total = self._new_spikes(mon, seen)
                img = np.clip(((t - t_off) // T_ms).astype(int), 0, c1 - c0 - 1) + c0
                np.add.at(counts[key], (img, i), 1)
                setattr(self, attr, total)
            if train:
                self.normalize()
            if verbose:
                print(f"\r  {c1}/{n} images", end="")
        if verbose:
            print()
        return counts

    def output_weights(self):
        W = np.zeros((self.nh, N_OUT))
        W[np.asarray(self.S_ho.i[:]), self._j_ho] = np.asarray(self.S_ho.w[:])
        return W

    def predict(self, counts):
        """WTA readout: most output spikes; ties (and the no-spike case) go to the largest total synaptic input."""
        drive = counts["hidden"] @ self.output_weights()
        drive = drive / (drive.max(axis=1, keepdims=True) + 1e-12)
        score = counts["output"] + 0.5 * drive          # drive is in [0, 1], so it only breaks ties
        tie_broken = (counts["output"].max(1) == 0) | ((counts["output"] == counts["output"].max(1, keepdims=True)).sum(1) > 1)
        return score.argmax(1), tie_broken

    def state(self):
        return dict(w_ih=np.asarray(self.S_ih.w[:]), w_hh=np.asarray(self.S_hh.w[:]),
                    w_ho=np.asarray(self.S_ho.w[:]), theta=np.asarray(self.hid.theta[:] / mV))


def spike_summary(snn, counts):
    """Per-inference spike counts and synaptic operations (SOPs = spikes × fan-out)."""
    fo = snn.fanout()
    s_in = counts["input"].mean()
    s_h = counts["hidden"].sum(1).mean()
    s_o = counts["output"].sum(1).mean()
    sops = s_in * fo["input"] + s_h * fo["hidden"] + s_o * fo["output"]
    return dict(input=s_in, hidden=s_h, output=s_o, total=s_in + s_h + s_o, sops=sops)


snn = SpikingNet(N_IN, CFG["n_hidden"], N_OUT, P, CFG, seed=CFG["seed"])
print(f"synapses: input->hidden {len(snn.S_ih)}, hidden recurrent exc {len(snn.S_hh)}, "
      f"hidden lateral inh {len(snn.S_inh_h)}, hidden->output {len(snn.S_ho)}, output inh {len(snn.S_inh_o)}")

## 4. Training: 20 epochs of online STDP
In each epoch the network sees all training images in a new random order, with plasticity on. After each epoch
plasticity is frozen and the **500-image test subset** is presented, which gives accuracy and
spikes per inference epoch by epoch.

In [ ]:
history = defaultdict(list)
t_train0 = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    te0 = time.time()
    perm = rng.permutation(len(X_train))
    tr_counts = snn.present(X_train[perm], y_train[perm], train=True)
    te_counts = snn.present(X_test, train=False)
    pred, tie = snn.predict(te_counts)
    acc = (pred == y_test).mean()
    tr_sp, te_sp = spike_summary(snn, tr_counts), spike_summary(snn, te_counts)
    history["epoch"].append(epoch); history["test_acc"].append(float(acc))
    history["tie_broken"].append(float(tie.mean()))
    for k, v in te_sp.items():
        history["test_" + k].append(v)
    history["train_hidden_spikes"].append(tr_sp["hidden"])
    history["theta_mean"].append(float(np.mean(snn.hid.theta[:] / mV)))
    print(f"epoch {epoch:2d} | test acc {acc*100:5.1f}% | spikes/inference: in {te_sp['input']:.0f}, "
          f"hidden {te_sp['hidden']:.1f}, out {te_sp['output']:.1f} | tie-break {tie.mean()*100:4.1f}% | "
          f"theta {history['theta_mean'][-1]:.2f} mV | {time.time()-te0:.0f}s")
last_train_counts, last_train_labels = tr_counts, y_train[perm]
print(f"training finished in {(time.time()-t_train0)/60:.1f} min")
np.savez("snn_trained_state.npz", **snn.state())

## 5. Results for the trained SNN

In [ ]:
final_counts = te_counts                     # test-set responses after epoch 20
snn_pred, snn_tie = snn.predict(final_counts)
snn_acc = (snn_pred == y_test).mean()
snn_sp = spike_summary(snn, final_counts)

# Diagnostic: Diehl & Cook-style vote over the hidden layer (each hidden neuron gets the label it fired most for
# during the last training epoch). This shows how much of the hidden representation the STDP-trained WTA readout uses.
resp = np.zeros((CFG["n_hidden"], N_OUT))
for c in range(N_OUT):
    resp[:, c] = last_train_counts["hidden"][last_train_labels == c].mean(0)
assign = resp.argmax(1)
votes = np.stack([final_counts["hidden"][:, assign == c].sum(1) / max((assign == c).sum(), 1)
                  for c in range(N_OUT)], 1)
vote_acc = (votes.argmax(1) == y_test).mean()

print(f"SNN test accuracy (output WTA layer, {len(y_test)} images): {snn_acc*100:.1f}%")
print(f"  decided by output spike count: {100*(1-snn_tie.mean()):.1f}% of images; tie-break on synaptic input: {100*snn_tie.mean():.1f}%")
print(f"  diagnostic, hidden-layer label vote: {vote_acc*100:.1f}%")
print("Spikes per inference: " + ", ".join(f"{k} {v:.1f}" for k, v in snn_sp.items() if k != "sops"))
print(f"Synaptic operations per inference: {snn_sp['sops']:.0f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ep = np.array(history["epoch"])
ticks = range(2, CFG["epochs"] + 1, 2) if CFG["epochs"] > 5 else ep
ax[0].plot(ep, np.array(history["test_acc"]) * 100, "o-", c=BLUE)
ax[0].axhline(10, ls=":", lw=1, c=GRAY); ax[0].text(ep[0], 4, "chance (10%)", color=INK, fontsize=8)
ax[0].set(xlabel="epoch", ylabel="test accuracy (%)", title="SNN test accuracy during online STDP training",
          xticks=ticks, ylim=(0, 100)); ax[0].grid(axis="y")
ax[1].bar(ep, history["test_hidden"], color=BLUE, edgecolor="white", linewidth=1, label="hidden")
ax[1].bar(ep, history["test_output"], bottom=history["test_hidden"], color=ORANGE, edgecolor="white", linewidth=1, label="output")
ax[1].set(xlabel="epoch", ylabel="spikes / inference", title="Hidden + output spikes per test image", xticks=ticks)
ax[1].legend(frameon=False, loc="lower right"); ax[1].grid(axis="y")
ax[2].plot(ep, history["theta_mean"], "o-", c=BLUE)
ax[2].set(xlabel="epoch", ylabel="mean θ (mV)", title="Adaptive threshold θ (homeostasis)", xticks=ticks); ax[2].grid(axis="y")
plt.tight_layout(); plt.show()

### 5.1 What the network learned
Each tile is one hidden neuron's input weights, reshaped to 34 × 34. STDP has turned them into digit
prototypes. The middle panel shows the hidden→output weights learned with the teacher signal, and the right panel
shows the learned recurrent excitatory weights.

In [ ]:
W_ih = np.zeros((N_IN, CFG["n_hidden"])); W_ih[np.asarray(snn.S_ih.i[:]), snn._j_ih] = snn.S_ih.w[:]
side = int(np.ceil(np.sqrt(CFG["n_hidden"])))
order = np.argsort(assign, kind="stable")
canvas = np.full((side * 34, side * 34), np.nan)
for n, h in enumerate(order):
    r, c = divmod(n, side)
    canvas[r * 34:(r + 1) * 34, c * 34:(c + 1) * 34] = W_ih[:, h].reshape(SENSOR)
W_ho = snn.output_weights()
W_hh = np.zeros((CFG["n_hidden"],) * 2); W_hh[np.asarray(snn.S_hh.i[:]), np.asarray(snn.S_hh.j[:])] = snn.S_hh.w[:]

fig = plt.figure(figsize=(17, 6))
ax0 = fig.add_subplot(1, 3, 1); ax0.imshow(canvas, cmap="Blues"); ax0.axis("off")
ax0.set_title("Input→hidden receptive fields (sorted by assigned digit)")
ax1 = fig.add_subplot(1, 3, 2); im = ax1.imshow(W_ho[order].T, aspect="auto", cmap="Blues")
ax1.set(xlabel="hidden neuron (sorted by assigned digit)", ylabel="output neuron (digit)", yticks=range(10),
        title="Hidden→output weights (teacher-guided STDP)"); plt.colorbar(im, ax=ax1)
ax2 = fig.add_subplot(1, 3, 3); im = ax2.imshow(W_hh[np.ix_(order, order)], cmap="Blues")
ax2.set(xlabel="post (sorted)", ylabel="pre (sorted)", title="Recurrent excitatory weights (STDP)"); plt.colorbar(im, ax=ax2)
plt.tight_layout(); plt.show()
print("hidden neurons assigned to each digit:", np.bincount(assign, minlength=10))

## 6. The equivalent ANN (PyTorch)
The ANN has the **same architecture**, 1156 → 100 → 10, with ReLU in place of LIF neurons. It is trained on the **same
2,000 intensity frames** (the frames the rate code encodes) for the **same 20 epochs**, using backpropagation
(Adam, cross-entropy), and tested on the same 500 images.

**Cost of one ANN inference**
* MACs (multiply-accumulates) for dense matrix multiplications: $1156 \cdot 100 + 100 \cdot 10 = 116{,}600$.
* "Spike equivalents": the number of non-zero activations (input pixels + hidden ReLUs + outputs). This is
  the closest ANN counterpart to a spike count, since each one is a value that has to be sent downstream.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.fc1, self.fc2 = nn.Linear(n_in, n_hidden), nn.Linear(n_hidden, n_out)

    def forward(self, x, return_hidden=False):
        h = torch.relu(self.fc1(x))
        out = self.fc2(h)
        return (out, h) if return_hidden else out


def train_ann(n_hidden, epochs, seed, verbose=False):
    torch.manual_seed(seed)
    Xtr, ytr = torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long)
    Xte, yte = torch.tensor(X_test), torch.tensor(y_test, dtype=torch.long)
    model = MLP(N_IN, n_hidden, N_OUT)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn, accs = nn.CrossEntropyLoss(), []
    g = torch.Generator().manual_seed(seed)
    for ep in range(epochs):
        model.train()
        for b in torch.randperm(len(Xtr), generator=g).split(32):
            opt.zero_grad(); loss = loss_fn(model(Xtr[b]), ytr[b]); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            accs.append((model(Xte).argmax(1) == yte).float().mean().item())
        if verbose:
            print(f"epoch {ep+1:2d} | loss {loss.item():.3f} | test acc {accs[-1]*100:.1f}%")
    with torch.no_grad():
        logits, h = model(Xte, return_hidden=True)
    pred = logits.argmax(1).numpy()
    cost = dict(
        macs=N_IN * n_hidden + n_hidden * N_OUT,
        activations=float((Xte != 0).sum(1).float().mean() + (h > 0).sum(1).float().mean() + N_OUT),
        nonzero_input=float((Xte != 0).sum(1).float().mean()),
        nonzero_hidden=float((h > 0).sum(1).float().mean()))
    # MACs left if zero activations are skipped (event-driven ANN)
    cost["sparse_macs"] = cost["nonzero_input"] * n_hidden + cost["nonzero_hidden"] * N_OUT
    return model, accs, pred, cost


ann, ann_hist, ann_pred, ann_cost = train_ann(CFG["n_hidden"], CFG["epochs"], CFG["seed"], verbose=True)
ann_acc = (ann_pred == y_test).mean()
print(f"\nANN test accuracy: {ann_acc*100:.1f}%  |  MACs/inference {ann_cost['macs']:,}  |  "
      f"non-zero activations/inference {ann_cost['activations']:.0f}")

## 7. Energy proxy: spikes vs activations

The main proxy is the **total spike count** per inference. To make the comparison with an ANN fair, the operations
behind those spikes are converted to energy using the standard 45 nm CMOS figures from Horowitz (ISSCC 2014),
as is common in SNN papers:

* SNN: every spike causes one **accumulate (AC)** per outgoing synapse, so $E_{SNN} = \text{SOPs} \times 0.9\,\text{pJ}$.
* ANN: every connection costs one **multiply-accumulate (MAC)**, so $E_{ANN} = \text{MACs} \times 4.6\,\text{pJ}$ (32-bit float).

Both estimates count synaptic operations only: neuron state updates and memory traffic are left out.

In [ ]:
E_AC, E_MAC = 0.9e-12, 4.6e-12   # joules per operation (45 nm, 32-bit float; Horowitz 2014)
snn_energy = snn_sp["sops"] * E_AC
ann_energy = ann_cost["macs"] * E_MAC

rows = [
    (f"Test accuracy ({len(y_test)} images)", f"{snn_acc*100:.1f} %", f"{ann_acc*100:.1f} %"),
    ("Spikes / non-zero activations per inference", f"{snn_sp['total']:.0f}", f"{ann_cost['activations']:.0f}"),
    ("  of which input layer", f"{snn_sp['input']:.0f}", f"{ann_cost['nonzero_input']:.0f}"),
    ("  of which hidden layer", f"{snn_sp['hidden']:.1f}", f"{ann_cost['nonzero_hidden']:.0f}"),
    ("  of which output layer", f"{snn_sp['output']:.1f}", f"{N_OUT}"),
    ("Synaptic ops per inference", f"{snn_sp['sops']:,.0f} AC", f"{ann_cost['macs']:,} MAC"),
    ("Estimated energy per inference", f"{snn_energy*1e9:.2f} nJ", f"{ann_energy*1e9:.2f} nJ"),
    (f"Total spikes over the {len(y_test)}-image test set", f"{snn_sp['total']*len(y_test):,.0f}", f"{ann_cost['activations']*len(y_test):,.0f}"),
]
w0 = max(len(r[0]) for r in rows)
print(f"{'':{w0}}  {'SNN (Brian2)':>16}  {'ANN (PyTorch)':>16}")
for r in rows:
    print(f"{r[0]:{w0}}  {r[1]:>16}  {r[2]:>16}")
print(f"\nEnergy ratio ANN / SNN = {ann_energy / snn_energy:.2f}x")

## 8. Energy-accuracy trade-off
An SNN's spike budget can be changed **after training** without retraining:
* **Presentation time** *T* (how long the image is shown), and
* **Maximum input rate** $r_{max}$ (the rate-code gain).

Both are swept on the trained network and the 500-image test subset. The training trajectory (one point
per epoch) is shown too. For the ANN, the hidden width is swept (10 to 200 units), since that is how an ANN
trades compute for accuracy.

In [ ]:
T_sweep = [10, 20, 35, 50, 75, 100, 150, 200]
rate_sweep = [10, 20, 35, 50, 75, 100, 150, 200]
sweeps = dict(T=[], rate=[])
for T_ms in T_sweep:
    c = snn.present(X_test, train=False, T_ms=T_ms)
    s = spike_summary(snn, c); a = (snn.predict(c)[0] == y_test).mean()
    sweeps["T"].append((T_ms, a, s["total"], s["sops"] * E_AC)); print(f"T = {T_ms:4.0f} ms  | acc {a*100:5.1f}% | spikes {s['total']:7.1f}")
for r in rate_sweep:
    c = snn.present(X_test, train=False, max_rate_hz=r)
    s = spike_summary(snn, c); a = (snn.predict(c)[0] == y_test).mean()
    sweeps["rate"].append((r, a, s["total"], s["sops"] * E_AC)); print(f"r_max = {r:4.0f} Hz | acc {a*100:5.1f}% | spikes {s['total']:7.1f}")

ann_sizes = [10, 25, 50, 100, 200]
ann_sweep = []
for nh in ann_sizes:
    _, _, pr, cst = train_ann(nh, CFG["epochs"], CFG["seed"])
    ann_sweep.append((nh, (pr == y_test).mean(), cst["activations"], cst["macs"] * E_MAC))
    print(f"ANN hidden={nh:3d} | acc {ann_sweep[-1][1]*100:5.1f}% | activations {cst['activations']:.0f} | MACs {cst['macs']:,}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ring = dict(markeredgecolor="white", markeredgewidth=1)
for key, lab, mk, col, unit in (("T", "SNN: presentation time T (10→200 ms)", "o-", BLUE, "ms"),
                                ("rate", "SNN: max input rate (10→200 Hz)", "s-", ORANGE, "Hz")):
    v, a, s_, e = map(np.array, zip(*sweeps[key]))
    ax[0].plot(s_, a * 100, mk, c=col, label=lab, **ring)
    ax[1].plot(e * 1e9, a * 100, mk, c=col, label=lab, **ring)
    for k in (0, len(v) - 1):        # label only the ends of each sweep
        ax[0].annotate(f"{v[k]:g} {unit}", (s_[k], a[k] * 100), textcoords="offset points",
                       xytext=(6, -12 if key == "T" else 6), fontsize=8, color=INK)
ax[0].plot(history["test_total"], np.array(history["test_acc"]) * 100, "^:", c=GRAY, lw=1,
           label="SNN: after each training epoch (1→20)")
nh, a, act, e = map(np.array, zip(*ann_sweep))
ax[0].plot(act, a * 100, "D--", c=AQUA, label="ANN: non-zero activations (width 10→200)", **ring)
ax[1].plot(e * 1e9, a * 100, "D--", c=AQUA, label="ANN: dense MACs (width 10→200)", **ring)
for x_, a_, n_ in zip(e * 1e9, a, nh):
    ax[1].annotate(f"{n_} hidden", (x_, a_ * 100), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color=INK)
star = dict(s=260, marker="*", edgecolor="white", linewidth=1.2, zorder=5)
ax[0].scatter([snn_sp["total"]], [snn_acc * 100], c=BLUE, label="trained SNN (T=100 ms, 100 Hz)", **star)
ax[0].scatter([ann_cost["activations"]], [ann_acc * 100], c=AQUA, label="equivalent ANN (100 hidden)", **star)
ax[1].scatter([snn_energy * 1e9], [snn_acc * 100], c=BLUE, **star)
ax[1].scatter([ann_energy * 1e9], [ann_acc * 100], c=AQUA, **star)
ax[0].set(xscale="log", xlabel="spikes (SNN) or non-zero activations (ANN) per inference",
          ylabel="test accuracy (%)", title="Accuracy vs spike count (energy proxy)", ylim=(0, 100))
ax[1].set(xscale="log", xlabel="estimated synaptic energy per inference (nJ, 45 nm)",
          ylabel="test accuracy (%)", title="Accuracy vs estimated energy", ylim=(0, 100))
for a_ in ax:
    a_.axhline(10, ls=":", lw=1, c=GRAY)
    a_.grid(which="major")
    a_.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
plt.tight_layout(); plt.savefig("accuracy_vs_spikes.png", dpi=150, bbox_inches="tight"); plt.show()

### 8.1 Confusion matrices

In [ ]:
def confusion(y, p):
    m = np.zeros((10, 10), int); np.add.at(m, (y, p), 1); return m

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a_, (name, pr, acc_) in zip(ax, (("SNN", snn_pred, snn_acc), ("ANN", ann_pred, ann_acc))):
    m = confusion(y_test, pr)
    a_.imshow(m, cmap="Blues")
    for i in range(10):
        for j in range(10):
            if m[i, j]:
                a_.text(j, i, m[i, j], ha="center", va="center", fontsize=7, color="w" if m[i, j] > m.max() / 2 else "k")
    a_.set(xlabel="predicted", ylabel="true", xticks=range(10), yticks=range(10), title=f"{name}: {acc_*100:.1f}%")
plt.tight_layout(); plt.show()

## 9. Summary of results
The cell below collects the headline numbers, saves them to `lab7_results.json`, and prints the key findings
in sentences. The discussion in Section 10 explains them.

In [ ]:
cheap = [r for r in sweeps["T"] + sweeps["rate"] if r[1] >= snn_acc - 0.02]
cheapest = min(cheap, key=lambda r: r[2]) if cheap else None
best = max(sweeps["T"] + sweeps["rate"], key=lambda r: r[1])
hid_frac_snn = float((final_counts["hidden"] > 0).sum(1).mean() / CFG["n_hidden"])
hid_frac_ann = ann_cost["nonzero_hidden"] / CFG["n_hidden"]
summary = {
    "SNN test accuracy (%)": round(float(snn_acc) * 100, 1),
    "ANN test accuracy (%)": round(float(ann_acc) * 100, 1),
    "SNN spikes per inference": round(float(snn_sp["total"]), 1),
    "SNN hidden+output spikes per inference": round(float(snn_sp["hidden"] + snn_sp["output"]), 1),
    "ANN non-zero activations per inference": round(ann_cost["activations"], 1),
    "SNN synaptic ops (AC) per inference": int(round(snn_sp["sops"])),
    "ANN MACs per inference": int(ann_cost["macs"]),
    "SNN energy per inference (nJ)": round(snn_energy * 1e9, 2),
    "ANN energy per inference (nJ)": round(ann_energy * 1e9, 2),
    "SNN accuracy per epoch (%)": [round(a * 100, 1) for a in history["test_acc"]],
    "ANN accuracy per epoch (%)": [round(a * 100, 1) for a in ann_hist],
}
for k, v in summary.items():
    print(f"{k:42s} {v}")
with open("lab7_results.json", "w") as f:
    json.dump(dict(config=CFG, summary=summary, sweeps=sweeps, ann_sweep=ann_sweep,
                   history={k: [float(x) for x in v] for k, v in history.items()}), f, indent=1, default=float)

se = np.sqrt(snn_acc * (1 - snn_acc) / len(y_test))   # binomial standard error of a test accuracy
print("\nKey findings")
print(f"* Accuracy: SNN {snn_acc:.1%} vs ANN {ann_acc:.1%} on the same {len(y_test)} test images "
      f"({(ann_acc - snn_acc) * 100:+.1f} points for the ANN). SNN accuracy after epoch 1: {history['test_acc'][0]:.1%}, "
      f"best epoch: {max(history['test_acc']):.1%}. With {len(y_test)} test images, one accuracy figure has a "
      f"standard error of about ±{se:.1%}, so differences of 1–2 points are noise.")
print(f"* Spikes: {snn_sp['total']:.0f} per inference, {snn_sp['input'] / snn_sp['total']:.1%} of them in the input layer. "
      f"Only {snn_sp['hidden']:.1f} hidden and {snn_sp['output']:.1f} output spikes per image.")
print(f"* Sparsity: {hid_frac_snn:.1%} of hidden LIF neurons fire for a test image, "
      f"against {hid_frac_ann:.1%} non-zero ReLUs in the ANN.")
print(f"* Operations: SNN {snn_sp['sops']:,.0f} accumulates vs ANN {ann_cost['macs']:,} MACs per inference; "
      f"estimated energy {snn_energy * 1e9:.0f} nJ vs {ann_energy * 1e9:.0f} nJ "
      f"(ANN/SNN = {ann_energy / snn_energy:.2f}x).")
if cheapest is not None:
    print(f"* Trade-off: the cheapest swept setting within 2 points of the default accuracy uses {cheapest[2]:.0f} spikes "
          f"({cheapest[2] / snn_sp['total']:.0%} of the default) at {cheapest[1]:.1%}. The most accurate setting reaches "
          f"{best[1]:.1%} with {best[2]:.0f} spikes.")

## 10. Discussion

**Accuracy.** The ANN is more accurate than the STDP-trained SNN, as expected. Backpropagation tunes every weight
to the classification loss. The SNN's hidden layer learns **without labels**: STDP plus competition gives each
hidden neuron one prototype (a typical way of writing one digit, see the receptive fields), and the labels reach
the network only through the local, Hebbian teacher signal on the hidden→output synapses. With 100 hidden neurons,
each digit gets about ten prototypes, so digits written in an unusual style match the wrong prototype. The
confusion matrix shows which digit pairs get confused: mostly digits whose motion-blurred event frames overlap
heavily. Accuracy rises quickly in the first epochs, while competition and the adaptive thresholds spread the
digits across the neurons (θ levels off once homeostasis reaches equilibrium), and then plateaus.

**Spike count as an energy proxy.** Nearly all SNN spikes are **input** spikes from the rate code. Lateral
inhibition keeps the hidden code extremely sparse: only a few hidden neurons fire per image, while many of
the ANN's ReLUs are non-zero for every image. Counting events, the SNN's internal (hidden + output) activity is
far below the ANN's. Counting operations, each input spike causes 100 accumulates (one per hidden neuron), so the SNN's
synaptic work scales with the input firing rate, while the ANN always performs all 116,600 MACs. The SNN emits
more events than the ANN has non-zero activations (a rate code needs several spikes per bright pixel), but each
event costs one accumulate instead of a multiply-accumulate (about 5× cheaper). So the estimated energy per
inference is lower for the SNN, by the factor printed in Section 7.

**The trade-off.** The accuracy vs spike count plot shows that the spike budget can be changed **after
training**. Shortening the presentation time or lowering the input rate cuts the spike count roughly in
proportion. The curve has a clear knee. Above it, accuracy is roughly flat: extra spikes add evidence but
change accuracy by no more than the sampling noise. Below it, accuracy falls to chance, because the
hidden neurons, whose thresholds were tuned during training at T = 100 ms and 100 Hz, no longer get enough
input spikes to fire within the presentation window. The best operating point is just above the knee, which
uses a fraction of the default spike budget (see the trade-off line in Section 9). An ANN cannot trade cost for
accuracy this way: its cost is set by its architecture, which is why the ANN curve is a sweep over hidden-layer
width (each point is a separately trained network).

**Limitations.**
1. The input is a rate code of accumulated event frames, which throws away the fine timing of the DVS events.
   A temporal code (time-to-first-spike) or feeding the events in directly would need far fewer spikes.
2. The energy model counts synaptic operations on a 45 nm process. Real neuromorphic hardware
   (Loihi, SpiNNaker) also pays for spike routing, neuron updates and memory access.
3. A 2,000-image training subset and 100 hidden neurons keep the Brian2 simulation to well under an hour.
   Diehl & Cook (2015) show that STDP networks gain accuracy steadily with more neurons and more training data.

## References
* G. Orchard, A. Jayawant, G. Cohen, N. Thakor, "Converting static image datasets to spiking neuromorphic datasets using saccades", *Frontiers in Neuroscience* 9:437, 2015.
* P. U. Diehl, M. Cook, "Unsupervised learning of digit recognition using spike-timing-dependent plasticity", *Frontiers in Computational Neuroscience* 9:99, 2015.
* M. Stimberg, R. Brette, D. Goodman, "Brian 2, an intuitive and efficient neural simulator", *eLife* 8:e47314, 2019.
* M. Horowitz, "Computing's energy problem (and what we can do about it)", *IEEE ISSCC*, 2014.